# v4 Pure-DIPF controller — how the model works

This notebook walks through the model in [`run1/dipf_controller.py`](run1/dipf_controller.py). It covers:

1. The simulation setup: road, ego vehicle and obstacle sizes
2. The **potential functions**: destination, lane boundary and obstacle
3. How the **force** is computed: the negative numerical gradient of the total potential
4. How the **acceleration** is computed: acceleration clamp, then jerk limit, then clamp again
5. How the **vehicle moves**: kinematic integration, `moveToXY` and `setSpeed`
6. **Boundary conditions and edge cases** in the code
7. Episode termination and the cost used by `optimize.py`
8. Mismatches between the code and the config or docs, found while writing this

The functions below are **copied from `dipf_controller.py`** (line numbers are cited) and use parameters from `run1/config.yaml`. That way the notebook runs **without SUMO or TraCI**. The controller itself cannot be imported without starting SUMO.

> **What v4 is:** the same controller as v3. The only change is that `optimize.py` tunes `(alpha, tau_y_base, sx, delta_lane, lambda_obs)` with **Differential Evolution** (DE/best/2 + exponential crossover). v3 used Optuna/TPE. The parameters reach the controller through environment variables such as `DIPF_ALPHA`. `tau_x_base` stays fixed.

In [ ]:
import math
from pathlib import Path
import numpy as np
import yaml
import matplotlib.pyplot as plt

CFG = yaml.safe_load(Path("run1/config.yaml").read_text())
SIM, EGO, DIPF = CFG["simulation"], CFG["ego"], CFG["dipf"]

# --- same names as dipf_controller.py ---
ROAD_LENGTH_M = SIM["road_length_m"]; LANES = SIM["lanes"]; LANE_WIDTH_M = SIM["lane_width_m"]
STEP_LENGTH = SIM["step_length"]
EGO_LENGTH, EGO_WIDTH = EGO["length_m"], EGO["width_m"]
EGO_MAX_SPEED, EGO_A_MAX = EGO["max_speed_mps"], EGO["max_accel_mps2"]
EGO_AY_MAX, EGO_VY_MAX = EGO["max_lateral_accel_mps2"], EGO["max_lateral_speed_mps"]
EGO_JERK_MAX, EGO_JERK_Y_MAX = EGO["max_jerk_mps3"], EGO["max_lateral_jerk_mps3"]
EGO_START_X = EGO["start_x"]
EGO_GOAL_X = ROAD_LENGTH_M - EGO["goal_x_offset_from_end_m"]
GOAL_LANE_INDEX = EGO["goal_lane_index"]

LAMBDA_OBS, ALPHA = DIPF["lambda_obs"], DIPF["alpha"]
TAU_X_BASE, TAU_Y_BASE = DIPF["tau_x_base"], DIPF["tau_y_base"]
SX, SY, DELTA_LANE = DIPF["sx"], DIPF["sy"], DIPF["delta_lane"]
W_DEST, W_OBS, W_LANE = DIPF["w_dest"], DIPF["w_obs"], DIPF["w_lane"]
DV_MAX = DIPF.get("dv_max", DIPF.get("dv_max_front", 10.0))
OBS_RADIUS_AHEAD, OBS_RADIUS_BEHIND = DIPF["obs_radius_ahead"], DIPF["obs_radius_behind"]
LATERAL_BAND, GRAD_EPS, Y_MARGIN = DIPF["lateral_band"], DIPF["grad_eps"], DIPF["y_margin"]
OBS_DIMS = CFG["obstacle_effective_dimensions"]

def clamp(x, lo, hi):
    return lo if x < lo else hi if x > hi else x

print({k: DIPF[k] for k in ["lambda_obs", "alpha", "tau_x_base", "tau_y_base", "sx", "sy", "delta_lane"]})

## 1. Geometry: ego size, road width, lanes

| Quantity | Value | Source |
|---|---|---|
| Ego length × width | **4.583 m × 1.748 m** | `ego.length_m`, `ego.width_m` (the `egoType` vType uses the same numbers) |
| Road length | 1000 m | `simulation.road_length_m` |
| Lanes × lane width (controller's assumption) | 3 × 3.5 m = **10.5 m**, y ∈ [−5.25, +5.25] | `lane_centers_y()` / `road_bounds_y()` (L290–300) |
| Goal | x = 985 m (1000 − 15), any lane (`success_any_lane: true`), tolerance ±1.5 m | `success_goal_reached()` L363 |
| Ego start | x = 2.5 m, lane index 2, departs at t = 20 s, 20 m/s | `ego.*` |

The controller assumes the road is **centered on y = 0**. Lane *i* has its center at `(i − (LANES−1)/2) · LANE_WIDTH` → **−3.5, 0, +3.5**.

Each obstacle has two sizes:
* **Effective (padded) size** from `obstacle_effective_dimensions`. It shapes the DIPF potential only.
* **Actual size** from SUMO (`getLength/getWidth`). It is used only for the logged `closest_dist` and the cost.

> ⚠️ **Lane-width mismatch (found while writing this notebook).** `generate_sumo.py` writes `laneWidth="3.5"` on the `<edge>`. SUMO's edge attribute is `width`, so netconvert ignores `laneWidth` and falls back to its default of **3.2 m**. The generated `straight3lane.net.xml` has lane centers at **y = −3.2, 0, +3.2**, so the real road spans **±4.8 m**. The controller still assumes ±5.25 m. The next cell checks this directly from the net file.

In [ ]:
def lane_centers_y():
    mid = (LANES - 1) / 2.0
    return [(i - mid) * LANE_WIDTH_M for i in range(LANES)]

def road_bounds_y():
    half = (LANES * LANE_WIDTH_M) / 2.0
    return -half, +half

print("Controller lane centers:", lane_centers_y(), " road edges:", road_bounds_y())

import re
net = Path("run1/straight3lane.net.xml").read_text()
for lid, shape in re.findall(r'<lane id="(e0_\d)"[^>]*shape="([^"]+)"', net):
    print(f"SUMO {lid}: shape={shape}   width attr present: {'width=' in re.search(r'<lane id=\"'+lid+r'\"[^>]*>', net).group(0)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.6))
yl, yr = road_bounds_y()
ax.axhspan(yl, yr, color="0.9", label="controller road (±5.25)")
ax.axhline(-4.8, color="C3", ls="--", lw=1, label="SUMO net road (±4.8)")
ax.axhline(4.8, color="C3", ls="--", lw=1)
for c in lane_centers_y():
    ax.axhline(c, color="0.6", ls=":", lw=1)
# ego at start lane center
x0, y0 = 20, lane_centers_y()[EGO["start_lane"]]
ax.add_patch(plt.Rectangle((x0 - EGO_LENGTH/2, y0 - EGO_WIDTH/2), EGO_LENGTH, EGO_WIDTH, color="m", label="ego"))
for i, (typ, xo) in enumerate([("bike", 35), ("car", 45), ("truck", 62)]):
    d = OBS_DIMS[typ]; yo = lane_centers_y()[i]
    ax.add_patch(plt.Rectangle((xo - d["length_m"]/2, yo - d["width_m"]/2), d["length_m"], d["width_m"],
                               fill=False, ec="C0", lw=1.5))
    ax.text(xo, yo, typ, ha="center", va="center", fontsize=8)
ax.set_xlim(0, 80); ax.set_ylim(-6, 6); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
ax.set_title("Road cross-section with ego and effective obstacle boxes (to scale)")
ax.legend(loc="upper right", fontsize=7, ncol=3); plt.tight_layout(); plt.show()

## 2. Potential functions

The ego sees a total potential (`calculate_total_potential`, L515):

$$U(x,y) = w_{dest}\,U_{dest} + w_{lane}\,U_{lane} + w_{obs}\sum_{o} U_{obs,o}$$

All three weights are 1.0 in the config.

### 2.1 Destination potential (L450): conic attractor

$$U_{dest} = \sqrt{s_x (x-x_d)^2 + s_y (y-y_d)^2}$$

Since **`sy = 0`**, this reduces to $\sqrt{s_x}\,|x - x_d|$. Consequences:
* The goal has **no lateral pull**. `goal_lane_index` has no effect on the motion, only on the potential's value.
* Its gradient has a **constant magnitude of $\sqrt{s_x}$** = 1.0 m/s² with the current `sx`, whatever the distance to the goal. Far from the goal, the destination term alone commands +1 m/s² forward.

### 2.2 Lane-boundary potential (L455): inverse-square walls

$$U_{lane} = \Delta_{lane}\left[\frac{1}{\max(\epsilon,\,y-y_L)^2} + \frac{1}{\max(\epsilon,\,y_R-y)^2}\right],\quad \epsilon = 10^{-3}$$

`y_L`, `y_R` are the **true road edges** (±5.25). `calculate_total_potential` unpacks the last two return values of `road_bounds_y_safe_for_ego()`, and those are the unshrunk edges. The shrunk `y_min/y_max` are computed but **never used**, even though that function's docstring says they shape U_lane.

### 2.3 Obstacle potential (L483): unified closing-speed form

$$U_{obs} = \lambda\,\frac{f(\Delta v)}{k'(\Delta x,\Delta y,\Delta v_x,\Delta v_y)},\qquad f(\Delta v)=\text{clip}\!\left(\frac{\Delta v}{\Delta v_{max}},0,1\right)$$

with $\Delta x = x_o - x_h$ and $\Delta v_x = v_{x,o} - v_{x,h}$ (obstacle minus host).

**Closing speed** (L464), with no front/rear/side classification:
$$\Delta v = \max\!\left(0,\; -\frac{\Delta x\,\Delta v_x + \Delta y\,\Delta v_y}{\sqrt{\Delta x^2+\Delta y^2}}\right)$$

**Velocity-dependent time scale** (L411):
$$\tau(\Delta v) = \tau_{base}\cdot\tfrac12\left[(1+\alpha\Delta v) + (\alpha\Delta v - 1)\tanh(\Delta v)\right]$$

This goes to $\tau_{base}\alpha\Delta v$ for large positive $\Delta v$ and to $\tau_{base}$ for large negative $\Delta v$. At $\Delta v = 0$ it equals $\tau_{base}/2$.

**Anisotropic "distance"** $k'$ (L422), with combined half-sizes $w = (W_{ego}+w_{eff})/2$ and $l = (L_{ego}+l_{eff})/2$:
$$k' = \begin{cases}\max\!\big(0.1,\,1-\tfrac{w}{|\Delta x|}\big)\cdot\sqrt{(\Delta x/\tau_x)^2 + (\Delta y/\tau_y)^2} & |\Delta y|\le l|\Delta x|/w\\[4pt]\max\!\big(0.1,\,1-\tfrac{l}{|\Delta y|}\big)\cdot\sqrt{\cdots} & \text{otherwise}\end{cases}\;+\;0.05$$

Note that the first (longitudinal) branch subtracts the **width** $w$ from $|\Delta x|$, and the lateral branch subtracts the **length** $l$ from $|\Delta y|$. That follows the formula as written in the docstring. Check it against the paper, because the usual choice is the reverse ($l$ with $\Delta x$).

In [ ]:
def U_dest(xh, yh, xd, yd):
    return math.sqrt(SX * (xh - xd) ** 2 + SY * (yh - yd) ** 2)

def U_lane(yh, y_left, y_right):
    eps = 1e-3
    return DELTA_LANE * ((1.0 / max(eps, yh - y_left)) ** 2 + (1.0 / max(eps, y_right - yh)) ** 2)

def tau_component(delta_v, base):
    scale = 0.5 * ((1.0 + ALPHA * delta_v) + (ALPHA * delta_v - 1.0) * math.tanh(delta_v))
    return base * scale

def k_prime(dx, dy, dvx, dvy, w_eff, l_eff):
    tau_x = tau_component(dvx, TAU_X_BASE)
    tau_y = tau_component(dvy, TAU_Y_BASE)
    eps = 1e-3
    tau_x = tau_x if abs(tau_x) > eps else eps
    tau_y = tau_y if abs(tau_y) > eps else eps
    abs_dx = abs(dx) if abs(dx) > eps else eps
    abs_dy = abs(dy) if abs(dy) > eps else eps
    if abs_dy <= (l_eff * abs_dx) / w_eff:
        term = (1.0 - (w_eff / abs_dx)) if abs_dx > w_eff else 0.1
        term = max(0.1, term)
    else:
        term = (1.0 - (l_eff / abs_dy)) if abs_dy > l_eff else 0.1
        term = max(0.1, term)
    kp = term * math.sqrt((dx / tau_x) ** 2 + (dy / tau_y) ** 2)
    return kp + 0.05

def relative_closing_speed(dx, dy, dvx, dvy):
    dist = math.hypot(dx, dy)
    if dist < 1e-6:
        return 0.0
    return max(0.0, -(dx * dvx + dy * dvy) / dist)

def U_obs_exact(xh, yh, vxh, vyh, xo, yo, vxo, vyo, w_eff=2.0, l_eff=5.0):
    dx, dy, dvx, dvy = xo - xh, yo - yh, vxo - vxh, vyo - vyh
    if dx > OBS_RADIUS_AHEAD or dx < -OBS_RADIUS_BEHIND:
        return 0.0
    if abs(dy) > LATERAL_BAND:
        return 0.0
    delta_v = relative_closing_speed(dx, dy, dvx, dvy)
    f_dv = clamp(delta_v / max(1e-6, DV_MAX), 0.0, 1.0)
    if f_dv <= 0.0:
        return 0.0
    w_combined = (EGO_WIDTH + w_eff) / 2.0
    l_combined = (EGO_LENGTH + l_eff) / 2.0
    return LAMBDA_OBS * f_dv / k_prime(dx, dy, dvx, dvy, w_combined, l_combined)

def calculate_total_potential(x, y, vx, vy, obstacles, goal_x, goal_y):
    y_left, y_right = road_bounds_y()          # == the last two values of road_bounds_y_safe_for_ego()
    total = W_DEST * U_dest(x, y, goal_x, goal_y)
    total += W_LANE * U_lane(y, y_left, y_right)
    for (xo, yo, vxo, vyo, w_eff, l_eff, *_r) in obstacles:
        total += W_OBS * U_obs_exact(x, y, vx, vy, xo, yo, vxo, vyo, w_eff=w_eff, l_eff=l_eff)
    return total

### 2.4 What the lane potential does laterally

The cell below plots $U_{lane}(y)$ and the lateral force $-\partial U_{lane}/\partial y$. The force is compared with the ±3 m/s² lateral acceleration clamp.

In [ ]:
yl, yr = road_bounds_y()
ys = np.linspace(yl + 0.9, yr - 0.9, 400)
Ul = np.array([U_lane(y, yl, yr) for y in ys])
Fy = -np.gradient(Ul, ys)

fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
axs[0].plot(ys, Ul); axs[0].set_yscale("log"); axs[0].set_title(r"$U_{lane}(y)$  (Δ_lane=%g)" % DELTA_LANE)
axs[1].plot(ys, Fy, label=r"$-\partial U_{lane}/\partial y$")
axs[1].axhspan(-EGO_AY_MAX, EGO_AY_MAX, color="C2", alpha=.15, label="±a_y,max")
axs[1].set_ylim(-60, 60); axs[1].set_title("lateral force from lane potential")
for a in axs:
    for c in lane_centers_y(): a.axvline(c, color="0.6", ls=":")
    a.set_xlabel("y [m]")
axs[1].legend(fontsize=8); plt.tight_layout(); plt.show()

for c in lane_centers_y():
    h = 1e-4
    f = -(U_lane(c + h, yl, yr) - U_lane(c - h, yl, yr)) / (2 * h)
    print(f"lane center y={c:+.2f}: lateral force from U_lane = {f:+8.2f} m/s^2  -> clamped to {clamp(f, -EGO_AY_MAX, EGO_AY_MAX):+.2f}")

**Takeaway:** with `delta_lane = 400`, the lane potential has a single minimum at the **road center (y = 0)**. At the outer lane centers (±3.5 m) its force is about 150 m/s². That saturates the lateral clamp, so the ego is pushed hard toward the middle lane. There is **no per-lane well**, so "staying in lane 0 or lane 2" is not a stable state of this field. The ego starts in lane index 2 (y = +3.5), so it swerves toward the center right after spawning.

### 2.5 The time scale τ(Δv)

In [ ]:
dv = np.linspace(-10, 10, 400)
fig, ax = plt.subplots(figsize=(6, 3.2))
for a in [0.5, ALPHA, 5.0]:
    s = 0.5 * ((1 + a * dv) + (a * dv - 1) * np.tanh(dv))
    ax.plot(dv, s, label=f"alpha={a:.2f}")
ax.axhline(0, color="k", lw=.5); ax.set_xlabel(r"$\Delta v$ [m/s]"); ax.set_ylabel(r"$\tau/\tau_{base}$")
ax.set_title("tau scale factor"); ax.legend(); plt.tight_layout(); plt.show()

s = 0.5 * ((1 + ALPHA * dv) + (ALPHA * dv - 1) * np.tanh(dv))
print(f"min tau/tau_base for alpha={ALPHA:.3f}: {s.min():.4f} at dv={dv[s.argmin()]:.2f}  (the code floors |tau| at 1e-3)")

### 2.6 Obstacle potential field around an approaching car

A car-sized obstacle sits at the origin of the plot. The ego drives at 20 m/s, and the obstacle drives at 12 m/s (ego closing at 8 m/s). The map shows $U_{obs}$ as a function of the ego position. The second panel uses the **same speed** for both vehicles.

In [ ]:
def obs_map(v_ego, v_obs, typ="car"):
    d = OBS_DIMS[typ]
    xs = np.linspace(-30, 20, 251); ys = np.linspace(-5, 5, 121)
    Z = np.zeros((len(ys), len(xs)))
    for j, yh in enumerate(ys):
        for i, xh in enumerate(xs):
            Z[j, i] = U_obs_exact(xh, yh, v_ego, 0.0, 0.0, 0.0, v_obs, 0.0, d["width_m"], d["length_m"])
    return xs, ys, Z

fig, axs = plt.subplots(1, 2, figsize=(12, 3.4))
for a, (ve, vo, t) in zip(axs, [(20, 12, "ego 20 m/s, obstacle 12 m/s (closing 8 m/s)"),
                                 (20, 20, "same speed (closing 0 m/s)")]):
    xs, ys, Z = obs_map(ve, vo)
    im = a.pcolormesh(xs, ys, Z, shading="auto", cmap="magma")
    a.add_patch(plt.Rectangle((-2.5, -1.05), 5, 2.1, fill=False, ec="c"))
    a.set_title(t, fontsize=9); a.set_xlabel("ego x relative to obstacle [m]"); a.set_ylabel("ego y rel. [m]")
    fig.colorbar(im, ax=a)
plt.tight_layout(); plt.show()

**Key property:** $U_{obs}$ is zero whenever the gap is constant or growing ($\Delta v \le 0$). A same-speed car directly ahead, or a car pulling away, produces **no repulsion at all**, however small the gap. Repulsion only appears once the ego is closing in. The region behind the obstacle (ego at negative relative x) lights up, because that is where the ego approaches it.

## 3. How the force is computed

The controller treats **force per unit mass = acceleration command = −∇U**. The gradient is **numerical** (`numerical_grad_U_total`, L527), using central differences in x and y with step `grad_eps = 0.05` m:

$$\frac{\partial U}{\partial x}\approx\frac{U(x+\epsilon,y)-U(x-\epsilon,y)}{2\epsilon},\qquad \frac{\partial U}{\partial y}\approx\frac{U(x,y+\epsilon)-U(x,y-\epsilon)}{2\epsilon}$$

Important details:
* The velocities `(vx, vy)` are **frozen** while differentiating. Only position is perturbed, so the obstacle force is the gradient of a velocity-parameterized field.
* The gradient is taken of the **whole sum**, so contributions from the goal, lane and all obstacles add linearly.
* The kinks at `k_prime`'s branch switch, the `max(0.1, ·)` floor, the `f(Δv)` cut-off at 0 and the obstacle-window edges (`obs_radius_*`, `lateral_band`) can all produce **gradient spikes**. The jerk limiter exists to absorb them (see §4).
* At the goal $x = x_d$, the conic $U_{dest}$ has a kink and the central difference returns 0. In practice the episode ends before that, because the goal tolerance is ±1.5 m.

In [ ]:
def numerical_grad_U_total(state, obstacles, xd, yd):
    x, y, vx, vy = state
    U = lambda xx, yy: calculate_total_potential(xx, yy, vx, vy, obstacles, xd, yd)
    eps = GRAD_EPS
    return ((U(x + eps, y) - U(x - eps, y)) / (2 * eps),
            (U(x, y + eps) - U(x, y - eps)) / (2 * eps))

def obstacle(typ, xo, yo, vxo, vyo=0.0):
    d = OBS_DIMS[typ]
    return (xo, yo, vxo, vyo, d["width_m"], d["length_m"], d["width_m"], d["length_m"])

goal_y = lane_centers_y()[GOAL_LANE_INDEX]
# Break the force into its parts for one example state
state = (100.0, 0.3, 20.0, 0.0)
obs = [obstacle("car", 115.0, 0.0, 12.0)]
parts = {
    "dest":  numerical_grad_U_total(state, [], EGO_GOAL_X, goal_y),
}
gx_all, gy_all = numerical_grad_U_total(state, obs, EGO_GOAL_X, goal_y)
# isolate dest/lane/obstacle
h = GRAD_EPS; x, y, vx, vy = state
gd = ((U_dest(x+h, y, EGO_GOAL_X, goal_y) - U_dest(x-h, y, EGO_GOAL_X, goal_y)) / (2*h), 0.0)
gl = (0.0, (U_lane(y+h, *road_bounds_y()) - U_lane(y-h, *road_bounds_y())) / (2*h))
go = (gx_all - gd[0] - gl[0], gy_all - gd[1] - gl[1])
for name, g in [("destination", gd), ("lane", gl), ("obstacle", go), ("TOTAL", (gx_all, gy_all))]:
    print(f"{name:12s}  F = -grad U = ({-g[0]:+8.3f}, {-g[1]:+8.3f})")

## 4. How the acceleration is computed

`ego_control_step` (L732–797) converts the force into a commanded acceleration in three stages:

| Stage | Code | Effect |
|---|---|---|
| 1. Acceleration clamp | `ax_raw = clamp(-dUx, ±EGO_A_MAX)`; `ay_raw = clamp(-dUy, ±EGO_AY_MAX)` | ±3.2 m/s² longitudinal, ±3.0 m/s² lateral. The **same bound applies to braking and accelerating**. |
| 2. Jerk limit | `ax = clamp(ax_raw, prev_ax ± EGO_JERK_MAX·dt)` | At most a 10 m/s³ × 0.1 s = **1.0 m/s² change per tick** |
| 3. Re-clamp | `clamp(ax, ±EGO_A_MAX)` | Keeps the bound in place explicitly |

`prev_ax`/`prev_ay` are stored as **function attributes** on `ego_control_step`. They are reset to 0 in `setup_ego_after_spawn()`. Going from 0 to full braking therefore takes about 4 ticks (0.4 s).

The x and y axes are clamped **independently** (a box, not a friction circle). The total acceleration can therefore reach $\sqrt{3.2^2+3.0^2}\approx 4.4$ m/s².

In [ ]:
class JerkLimiter:
    def __init__(self): self.prev_ax = self.prev_ay = 0.0
    def __call__(self, dUx, dUy, dt=STEP_LENGTH):
        ax_raw = clamp(-dUx, -EGO_A_MAX, EGO_A_MAX); ay_raw = clamp(-dUy, -EGO_AY_MAX, EGO_AY_MAX)
        ax = clamp(ax_raw, self.prev_ax - EGO_JERK_MAX * dt, self.prev_ax + EGO_JERK_MAX * dt)
        ay = clamp(ay_raw, self.prev_ay - EGO_JERK_Y_MAX * dt, self.prev_ay + EGO_JERK_Y_MAX * dt)
        ax = clamp(ax, -EGO_A_MAX, EGO_A_MAX); ay = clamp(ay, -EGO_AY_MAX, EGO_AY_MAX)
        self.prev_ax, self.prev_ay = ax, ay
        return ax, ay

# Response to a one-tick gradient spike and to a sustained braking demand
lim = JerkLimiter()
demand = np.r_[np.full(5, -1.0), 50.0, np.full(5, -1.0), np.full(15, 20.0)]   # dUx sequence
out = [lim(d, 0.0)[0] for d in demand]
t = np.arange(len(demand)) * STEP_LENGTH
plt.figure(figsize=(8, 3))
plt.step(t, np.clip(-demand, -6, 6), where="post", label="-dU/dx (clipped for display)", alpha=.6)
plt.step(t, out, where="post", label="commanded a_x")
plt.axhline(-EGO_A_MAX, ls=":", c="k"); plt.axhline(EGO_A_MAX, ls=":", c="k")
plt.xlabel("t [s]"); plt.ylabel("m/s²"); plt.legend(); plt.title("acceleration clamp + jerk limit"); plt.tight_layout(); plt.show()

## 5. How the vehicle movement is simulated

For each SUMO step (`run_simulation`, L1117), the controller does the following:

1. `traci.simulationStep()` advances SUMO by `dt = 0.1 s`. All background traffic (bike/auto/car/bus/truck flows with the SL2015 sublane model) moves by SUMO's own car-following and lane-change logic.
2. It checks `traci.simulation.getCollisions()`. If the ego is involved, the episode ends as **collision**. SUMO runs with `--collision.action warn`, so it only reports collisions and does not remove vehicles.
3. The first time the ego appears (t ≈ 20 s), `setup_ego_after_spawn()` runs `setSpeedMode(0)` and `setLaneChangeMode(0)`. From then on **SUMO applies no safety checks and makes no lane changes for the ego**.
4. `ego_control_step`:
   * reads the state from SUMO: `x, y = getPosition`, `vx = getSpeed`, `vy = getLateralSpeed`
   * builds the obstacle list: every vehicle in `[x−20, x+50] × [y±8]`
   * computes the gradient, then the acceleration (§3–4)
   * integrates with **constant-acceleration kinematics**:
     $$x_{new} = x + v_x\,dt + \tfrac12 a_x dt^2,\qquad y_{new} = y + v_y\,dt + \tfrac12 a_y dt^2$$
     $$v_{x,apply} = \text{clip}(v_x + a_x dt,\;0,\;34),\qquad v_{y,apply} = \text{clip}(v_y + a_y dt,\;\pm 3)$$
   * runs a hard **road-footprint check** (`ego_within_road`). If it fails, the step counts as a collision.
   * **teleports** the ego with `traci.vehicle.moveToXY(ego, "e0", lane_i, x_new, y_new, angle=INVALID, keepRoute=2)`. `keepRoute=2` allows any position, including between lanes. The heading is derived from the displacement.
   * `traci.vehicle.setSpeed(ego, vx_apply)`

So SUMO serves only as the **traffic simulator and position store**. The ego's dynamics are a point-mass double integrator that lives inside the controller.

Things to keep in mind:
* **Position and velocity use slightly different integrators.** Position uses the *measured* `vx` plus ½a·dt². Speed uses `vx + a·dt`. The next step then reads `vx` back from SUMO, not `vx_apply`.
* **`vy_apply` is computed but never sent to SUMO.** The lateral state carries over only through `getLateralSpeed()`. If SUMO reports ~0 for a `moveToXY`-driven vehicle, the lateral motion loses its memory: each step moves only ½·a_y·dt² ≤ 0.015 m. Check the `vy` column in `trajectory.csv` to see which case applies.
* The speed is clamped at **0**, so the ego never reverses. The position integration is not clamped the same way, however: with `vx≈0` and `ax<0`, `x_new` can move slightly backwards.

The cell below is a **SUMO-free toy rollout** of the same loop with one slower car ahead in the middle lane. It shows the mechanics only and is not a substitute for a SUMO run.

In [ ]:
def ego_within_road(x, y):
    y_left, y_right = road_bounds_y()
    if (y - EGO_WIDTH / 2) < y_left or (y + EGO_WIDTH / 2) > y_right: return False
    if (x - EGO_LENGTH / 2) < 0.0 or (x + EGO_LENGTH / 2) > ROAD_LENGTH_M: return False
    return True

def toy_rollout(T=20.0, y0=lane_centers_y()[EGO["start_lane"]], v0=20.0, lead=("car", 60.0, 0.0, 12.0)):
    dt = STEP_LENGTH; lim = JerkLimiter()
    x, y, vx, vy = EGO_START_X + 5, y0, v0, 0.0
    typ, xo, yo, vxo = lead
    log = []
    for k in range(int(T / dt)):
        xo += vxo * dt                                      # "SUMO" moves the obstacle
        obs = [obstacle(typ, xo, yo, vxo)] if (x - 20 <= xo <= x + 50 and abs(yo - y) <= 8) else []
        dUx, dUy = numerical_grad_U_total((x, y, vx, vy), obs, EGO_GOAL_X, goal_y)
        ax, ay = lim(dUx, dUy, dt)
        x_new = x + vx * dt + 0.5 * ax * dt**2
        y_new = y + vy * dt + 0.5 * ay * dt**2
        vx = clamp(vx + ax * dt, 0.0, EGO_MAX_SPEED)
        vy = clamp(vy + ay * dt, -EGO_VY_MAX, EGO_VY_MAX)   # assumes SUMO reports back vy_apply
        if not ego_within_road(x_new, y_new):
            log.append((k*dt, x_new, y_new, vx, vy, ax, ay, xo)); print("road-departure COLLISION at t=%.1f" % (k*dt)); break
        x, y = x_new, y_new
        log.append((k * dt, x, y, vx, vy, ax, ay, xo))
    return np.array(log)

L = toy_rollout()
fig, axs = plt.subplots(1, 3, figsize=(14, 3.2))
axs[0].plot(L[:, 1], L[:, 2], label="ego"); axs[0].plot(L[:, 7], np.zeros(len(L)), "--", label="lead car")
for c in lane_centers_y(): axs[0].axhline(c, c="0.7", ls=":")
axs[0].set_xlabel("x"); axs[0].set_ylabel("y"); axs[0].legend(); axs[0].set_title("path")
axs[1].plot(L[:, 0], L[:, 3], label="vx"); axs[1].plot(L[:, 0], L[:, 4], label="vy"); axs[1].legend(); axs[1].set_title("velocity")
axs[2].plot(L[:, 0], L[:, 5], label="ax"); axs[2].plot(L[:, 0], L[:, 6], label="ay"); axs[2].legend(); axs[2].set_title("commanded accel")
for a in axs[1:]: a.set_xlabel("t [s]")
plt.tight_layout(); plt.show()
def closest_dist(dx, dy, sum_length, sum_width):     # L560: bounding-box clearance
    return math.hypot(max(0.0, abs(dx) - sum_length / 2), max(0.0, abs(dy) - sum_width / 2))
cd = [closest_dist(r[7] - r[1], 0.0 - r[2], EGO_LENGTH + OBS_DIMS["car"]["length_m"], EGO_WIDTH + OBS_DIMS["car"]["width_m"]) for r in L]
k = int(np.argmin(cd))
print(f"min body clearance to lead car: {cd[k]:.2f} m at t={L[k,0]:.1f}s (ego y={L[k,2]:+.2f})")
print(f"lateral oscillation: y range over run = [{L[:,2].min():+.2f}, {L[:,2].max():+.2f}] m")

In [ ]:
# How strong is the obstacle force compared with the constant +1 m/s^2 goal pull?
rows = []
for gap in [30, 20, 10, 6, 4, 3]:            # centre-to-centre distance, same lane, closing 8 m/s
    o = [obstacle("car", 100 + gap, 0.0, 12.0)]
    g  = numerical_grad_U_total((100, 0, 20, 0), o, 1e9, 0)
    g0 = numerical_grad_U_total((100, 0, 20, 0), [], 1e9, 0)
    rows.append((gap, U_obs_exact(100, 0, 20, 0, 100 + gap, 0, 12, 0, 2.1, 5.0), -(g[0] - g0[0])))
print(" centre gap | U_obs  | obstacle force a_x")
for gap, u, f in rows: print(f"   {gap:5.1f} m | {u:6.3f} | {f:+7.3f} m/s^2")

**What the toy rollout shows** (SUMO-free, one lead car, current `config.yaml` values):

* **The lateral motion is an undamped oscillation.** $U_{lane}$ is a spring toward y = 0, and nothing in the model damps $v_y$. The obstacle term depends on velocity, but the lane term does not. With the ±3 m/s² clamp, the ego swings bang-bang across all three lanes. (In SUMO this depends on whether `getLateralSpeed()` returns the implied $v_y$; see §5.)
* **The obstacle repulsion is weak compared with the goal pull.** With `lambda_obs = 1`, $U_{obs} \le 20$, and the force stays well under the destination's +1 m/s² until the bodies nearly touch. The ego never brakes for the lead car. In this run its body **overlaps the lead car** (clearance 0 m at t ≈ 4.6 s), which SUMO would report as a collision. `lambda_obs` (and `DV_MAX`) are the knobs that set this balance. They are also exactly what `optimize.py` searches over.

## 6. Boundary conditions and edge cases in the code

### Road and goal boundaries
| Case | Where | Handling |
|---|---|---|
| Ego body leaves the road (laterally or longitudinally) | `ego_within_road` L327, checked at L810 | **Counted as a COLLISION and the episode ends.** The position is *never* clamped back onto the road, so the acceleration limits are never violated by a clamp. The checked edges are x ∈ [0, 1000] and y ∈ ±5.25 (see the lane-width mismatch in §1). |
| Soft lateral wall | `U_lane` | Inverse-square repulsion from the true edges. `max(eps, ·)` with eps = 1e-3 prevents division by zero, which also makes the potential **flat (though huge) beyond the edge**. |
| Shrunk "safe" bounds | `road_bounds_y_safe_for_ego` L302 | Computed (`y_margin`, half-width, 0.05 buffer), but the shrunk values are **unused** |
| Reached the goal | `success_goal_reached` L363 | Triggers when \|x − 985\| ≤ 1.5 in any lane. The ego is snapped to `x = EGO_GOAL_X` and `setSpeed(0)` is called, returning DONE. That snap still runs through `moveToXY` and can itself return COLLISION. |
| Overshooting the goal | — | The goal check only tests \|x − x_d\|. If one step jumped past the 3 m window, `U_dest` would pull the ego back, but the speed is clamped ≥ 0. At 34 m/s × 0.1 s = 3.4 m per step, a jump past the window is possible in principle. |
| End of road | `ego_within_road` | x + L/2 > 1000 counts as a collision. The goal at 985 m normally comes first. |

### Numerical guards in the potentials
| Case | Where | Guard |
|---|---|---|
| τ ≈ 0 | `k_prime` L431 | \|τ\| floored at 1e-3, keeping its sign. For α ≈ 1.64, τ stays positive (see §2.5). |
| Δx or Δy ≈ 0 | `k_prime` | \|Δx\|, \|Δy\| floored at 1e-3 (used in the branch test and the `term`) |
| Bodies overlapping in the potential (\|Δx\| ≤ w or \|Δy\| ≤ l) | `k_prime` L439/442 | `term = max(0.1, …)`, so the potential is capped rather than singular |
| k' = 0 | L447 | `+ 0.05` floor. The maximum obstacle potential is therefore finite: U ≤ λ/0.05 = 20λ. |
| Coincident centers | `relative_closing_speed` L464 | dist < 1e-6 returns Δv = 0, so **there is no repulsion at all** |
| Separating or same-speed obstacle | L464, L502 | Δv clipped to ≥ 0 and f(Δv) to [0, 1], giving zero potential |
| Obstacle outside the window | `U_obs_exact` L496 and `build_obstacle_list` | Ignored if Δx > 50, Δx < −20 or \|Δy\| > 8. The potential switches on abruptly at the window edge, which causes a gradient step. |
| Unknown vehicle type | `get_obstacle_dimensions` L638 | Warns once, then uses `default`, then a hard-coded 2.1 × 5.0 |
| TraCI getters fail | `build_obstacle_list` | `vyo` falls back to 0, the type to "default", and the actual size to the effective size |

### Control and dynamics limits
| Case | Handling |
|---|---|
| Gradient spike | Acceleration clamp, then jerk limit (1 m/s² per tick), then clamp again |
| Speed limits | `vx_apply ∈ [0, 34]` (no reversing); `vy_apply ∈ [−3, 3]` |
| Jerk state across episodes | Reset in `setup_ego_after_spawn`, with a module-level default of 0 |

### Episode termination
| Status | Condition |
|---|---|
| `collision` | SUMO's collision detector involves the ego, **or** a road departure, **or** `moveToXY` raises a TraCIException containing "collision". A collision row with `closest_dist = 0` and NaNs is appended. |
| `success` | Goal reached |
| `stuck` | A timeout **and** some 8 s window with mean vx < 0.35 m/s and < 2 m progress |
| `timeout` | Reached `simulation.end` = 3600 s |
| Exception | Any other `moveToXY` failure raises a RuntimeError. The history is dumped to `crash_debug_hist.csv` and the error is re-raised, never swallowed. |

## 7. Episode cost (what `optimize.py` minimizes)

`compute_episode_cost` (L920):

$$d = \min(\text{closest\_dist},\ \text{road\_edge\_dist},\ 0.1),\quad C_{coll} = \frac{e^{1-d/0.1}}{e-1}$$
$$C_{dist} = 1 - \text{progress},\quad C_{time} = \min\!\left(1,\ \frac{(t_f-t_0)\,v_{max}}{3\cdot\text{progress}\cdot L_{road}}\right)$$
$$\boxed{C = 0\cdot C_{coll} + 20\,C_{dist} + C_{time}}$$

> ⚠️ The docstring says `C = 25·C_coll + 5·C_dist + C_time` with a 0.5 m cap on d. The **code uses a 0.1 m cap and gives the collision term weight 0**. Right now the optimizer only rewards progress and time. A collision is penalized only because it ends the episode early, which lowers progress.

## 8. Mismatches found (summary)

1. **Lane width:** SUMO uses 3.2 m because `laneWidth` is not a valid edge attribute (it should be `width`). The controller assumes 3.5 m. As a result, the lateral road-departure check and `U_lane` use edges at ±5.25 m while SUMO's road ends at ±4.8 m, and the controller's lane centers (±3.5) differ from SUMO's (±3.2).
2. **Config keys that are never read** (the code falls back to defaults that happen to match today):
   * `movement.move_match_threshold` and `move_keep_route`: the code reads `match_threshold` and `keep_route`.
   * `movement.lateral_resolution_m` and `debug_interval_sec`: the code reads `simulation.lateral_resolution` and `simulation.debug_interval`.
   * `movement.x_min_safe` and `x_max_margin_m`: not used anywhere.
   * `dipf.dv_max` does not exist, so `dv_max_front` = 10 is used.
3. **The shrunk safe bounds are unused.** `U_lane` uses the true edges, contradicting the docstring.
4. **`sy = 0`**, so the goal lane has no effect on the motion.
5. **The lane potential has one global minimum at the road center.** The outer lanes are not equilibria, and with no damping on $v_y$ the ego oscillates laterally.
6. **The obstacle force is weak when `lambda_obs = 1`.** It is below 0.2 m/s² even at a 4 m center gap, compared with the goal's constant +1 m/s², so the ego does not brake for slower traffic.
7. **Obstacles at the same speed or pulling away produce zero repulsion**, however close they are.
8. **The cost docstring and code disagree.** The collision weight is 0.
9. **`vy_apply` is never applied.** Lateral velocity continuity depends on what `getLateralSpeed()` returns after `moveToXY`.